# Tour Stop Order Editor - Standalone

This notebook allows you to reorder stops within tours after VRP optimization.

## How to Use

1. **Setup**: Mount Drive and load modules
2. **Load Data**: Load your VRP results (from JSON or main notebook)
3. **View Tours**: See current tour structure
4. **Export**: Export tours to CSV for editing
5. **Edit**: Edit CSV in Excel/Google Sheets (change `Tour Stop Index`)
6. **Import**: Import edited CSV to update tour order
7. **Continue**: Use updated tours in your workflow

## Important Notes

- Lower `Tour Stop Index` = earlier in tour
- Keep activities with same stop index together
- Times may need recalculation after reordering


In [ ]:
# @title Step 1: Setup and Mount Drive

from google.colab import drive
import sys
import os

# Mount Google Drive
drive.mount('/content/drive')

# Set path to tourplanning_modules folder
# ⚠️ UPDATE THIS PATH to match your folder structure
module_path = '/content/drive/MyDrive/Colab Notebooks/tourplanning_modules'
if module_path not in sys.path:
    sys.path.insert(0, module_path)

# Add tour_editor to path
tour_editor_path = os.path.join(module_path, 'tour_editor')
if tour_editor_path not in sys.path:
    sys.path.insert(0, tour_editor_path)

print("✅ Setup complete!")
print(f"   Module path: {module_path}")
print(f"   Tour editor path: {tour_editor_path}")


In [ ]:
# @title Step 2: Import Required Modules

import pandas as pd
import json
from tour_editor.module_tour_editor import (
    export_tours_for_editing,
    import_tours_from_csv,
    get_tour_stops_dataframe,
    create_simple_editor_interface
)

print("✅ Modules imported successfully!")


## Step 3: Load Your Data

You have two options:

**Option A**: Load from a saved JSON file (if you saved VRP results)
**Option B**: Load from variables (if running after main TourPlanning_Script)

Choose one option below.


In [ ]:
# @title Option A: Load from JSON File

# ⚠️ UPDATE THIS PATH to your VRP response JSON file
vrp_response_file = '/content/drive/MyDrive/Colab Notebooks/kemmler/2025-12-15/vrp_response.json'  # @param {type:"string"}

# Load VRP response
with open(vrp_response_file, 'r') as f:
    vrp_response_json = json.load(f)

# Load geocoded data (needed for customer names, addresses)
# ⚠️ UPDATE THIS PATH to your geocoded CSV file
geocoded_file = '/content/drive/MyDrive/Colab Notebooks/kemmler/2025-12-15/geocoded_data.csv'  # @param {type:"string"}
df_geocoded = pd.read_csv(geocoded_file)

print("✅ Data loaded successfully!")
print(f"   Tours: {len(vrp_response_json.get('tours', []))}")
print(f"   Geocoded orders: {len(df_geocoded)}")


In [ ]:
# @title Option B: Use Variables from Main Notebook

# If you're running this after TourPlanning_Script.ipynb,
# the variables should already be available.
# Just run this cell to verify they exist.

try:
    # Check if variables exist
    if 'vrp_response_json' in globals():
        print("✅ vrp_response_json found")
        print(f"   Tours: {len(vrp_response_json.get('tours', []))}")
    else:
        print("❌ vrp_response_json not found")
        print("   Please run Option A or ensure you've run the main notebook first")
    
    if 'df_geocoded' in globals():
        print("✅ df_geocoded found")
        print(f"   Orders: {len(df_geocoded)}")
    else:
        print("❌ df_geocoded not found")
        print("   Please run Option A or ensure you've run the main notebook first")
        
except Exception as e:
    print(f"❌ Error: {str(e)}")
    print("   Please use Option A to load from files")


## Step 4: View Current Tour Structure

View the current stop order for all tours.


In [ ]:
# @title View Current Tours

# Display tour structure
create_simple_editor_interface(vrp_response_json, None, df_geocoded)


## Step 5: Export Tours for Editing

Export all tours to CSV. You can then download and edit this file.


In [ ]:
# @title Export Tours to CSV

# Set output path
# ⚠️ UPDATE THIS PATH to where you want to save the CSV
base_date_str = "2025-12-15"  # @param {type:"string"}
client_name = "Kemmler"  # @param {type:"string"}

output_folder_path = f'/content/drive/MyDrive/Colab Notebooks/{client_name.lower()}/{base_date_str}'
export_path = os.path.join(output_folder_path, f'{base_date_str}_tours_for_editing.csv')

# Create output folder if it doesn't exist
os.makedirs(output_folder_path, exist_ok=True)

# Export tours
print("Exporting tours...")
tour_df = export_tours_for_editing(
    vrp_response_json,
    df_geocoded,
    export_path
)

print(f"\n📥 Download the file from:")
print(f"   {export_path}")
print(f"\n📝 Editing Instructions:")
print(f"   1. Open the CSV in Excel or Google Sheets")
print(f"   2. Edit the 'Tour Stop Index' column to change order")
print(f"      - Lower numbers = earlier in tour")
print(f"      - 0 = first stop, 1 = second stop, etc.")
print(f"   3. Keep activities with same stop index together")
print(f"   4. Save the file")
print(f"   5. Upload it back to the same location")
print(f"   6. Run the import cell below")


## Step 6: Import Edited Tours

After editing the CSV file, upload it back and import it here.


In [ ]:
# @title Import Edited CSV

# Path to edited CSV (should be the same as export path)
edited_csv_path = export_path  # @param {type:"string"}

# Check if file exists
if not os.path.exists(edited_csv_path):
    print(f"❌ File not found: {edited_csv_path}")
    print("   Please make sure you've uploaded the edited CSV file.")
else:
    try:
        # Import edited tours
        print("Importing edited tours...")
        updated_vrp_response = import_tours_from_csv(
            edited_csv_path,
            vrp_response_json
        )
        
        # Update the variable
        vrp_response_json = updated_vrp_response
        
        print("\n✅ Tours imported successfully!")
        print("   Tour order has been updated.")
        print("\n📋 Next Steps:")
        print("   1. Continue with your workflow using the updated vrp_response_json")
        print("   2. Run results merging with the new order")
        print("   3. Export and visualize updated tours")
        
    except Exception as e:
        print(f"❌ Error importing tours: {str(e)}")
        print("\nCommon issues:")
        print("   - CSV format doesn't match export format")
        print("   - Missing required columns")
        print("   - Invalid Tour Stop Index values")
        import traceback
        traceback.print_exc()


## Step 7: Verify Updated Tours

View the updated tour structure to verify your changes.


In [ ]:
# @title View Updated Tours

# Display updated tour structure
create_simple_editor_interface(vrp_response_json, None, df_geocoded)

print("\n✅ If the order looks correct, you can now:")
print("   1. Save the updated vrp_response_json to a file")
print("   2. Continue with results merging in your main notebook")
print("   3. Export and visualize the updated tours")


## Step 8: Save Updated VRP Response (Optional)

Save the updated VRP response to a file for later use.


In [ ]:
# @title Save Updated VRP Response

save_updated_response = True  # @param {type:"boolean"}

if save_updated_response:
    # Save path
    save_path = os.path.join(output_folder_path, f'{base_date_str}_vrp_response_edited.json')
    
    # Save to file
    with open(save_path, 'w') as f:
        json.dump(vrp_response_json, f, indent=2)
    
    print(f"✅ Updated VRP response saved to:")
    print(f"   {save_path}")
    print("\nYou can now:")
    print("   1. Use this file in your main workflow")
    print("   2. Continue with results merging")
    print("   3. Export and visualize updated tours")
else:
    print("Skipping save. Updated vrp_response_json is available in memory.")


## Next Steps

After editing tours, you can:

1. **Continue in Main Notebook**: Use the updated `vrp_response_json` variable
2. **Run Results Merging**: Merge results with the new tour order
3. **Export & Visualize**: Create maps and export files with updated order

### To Continue in Main Notebook:

```python
# In your TourPlanning_Script.ipynb, after Step 5:
# The vrp_response_json variable will have the updated order
# Continue with Step 6: Results Merging
merged_df, df_unassigned_jobs = module_results.merge_results(
    vrp_response_json, df_geocoded, output_folder_path, depots_path
)
```

---

**Note**: Times and distances may need recalculation after reordering. The module preserves original times but doesn't automatically recalculate them.
